In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/src/LorentzianSimplexSolver`


In [2]:
# ------------------------------------------------------------
# 1. Precision choice (user-controlled)
# ------------------------------------------------------------
const ScalarT = Float64
#const ScalarT = BigFloat

if ScalarT === BigFloat
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(256)
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(1e-8)
end

# ------------------------------------------------------------
# 2. Read simplices
# ------------------------------------------------------------
simplices = [[2,3,4,6,7],[1,3,4,6,7],[1,2,4,6,7],[1,2,3,6,7],[1,2,3,4,7],[2,3,5,6,8],[1,3,5,6,8],[1,2,5,6,8],[1,2,3,6,8],[1,2,3,5,8],[2,4,5,6,9],[1,4,5,6,9],[1,2,5,6,9],[1,2,4,6,9],[1,2,4,5,9],[3,4,5,6,10],[1,4,5,6,10],[1,3,5,6,10],[1,3,4,6,10],[1,3,4,5,10],[3,4,5,6,11],[2,4,5,6,11],[2,3,5,6,11],[2,3,4,6,11],[2,3,4,5,11]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

Nverts = length(all_vertices)

# ------------------------------------------------------------
# 3. Read vertex coordinates
# ------------------------------------------------------------
vertex_coords = Dict{Int, Vector{ScalarT}}()    

coords_lines = [
"0, 0, 0, 0",
"0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
"0, 0, 0, -3.398088489694245",
"-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
"0, 0, -2.942830956382712, -1.6990442448471226",
"-0.068, -0.27, -0.5, -1.3",
"-0.06165622828269508, -0.7476319083813029, -0.49237746085102824, -1.6192353958776982",
"-0.013600000000000001, -0.6089055267050423, -0.8847549217020566, -1.6192353958776982",
"-0.05471634704156723, -0.6634799624836852, -1.2905133626305172, -1.3266576479993824",
"-0.05996263164968173, -0.18743250119993968, -0.8604521717798855, -1.5747576871100133",
"-0.0423034122998675, -0.5046526423930288, -1.0289336294868097, -2.25080216236223"
]

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [3]:
# ------------------------------------------------------------
# 4. Build geometry
# ------------------------------------------------------------
datasets = LorentzianSimplexSolver.GeometryTypes.GeometryDataset{ScalarT}[]

for (s, simplex) in enumerate(simplices)
    bdypoints = [vertex_coords[v] for v in simplex]
    # println("Processing simplex $s with vertices $(simplex)...")        
    ds = LorentzianSimplexSolver.GeometryPipeline.run_geometry_pipeline(bdypoints)
    push!(datasets, ds)
end

geom = LorentzianSimplexSolver.GeometryTypes.GeometryCollection(datasets);

In [4]:
# ------------------------------------------------------------
# 5. Connect simplices + face matching + gauge fixing
# ------------------------------------------------------------
if ns > 1
    LorentzianSimplexSolver.KappaOrientation.fix_kappa_signs!(simplices, geom)

    conn = LorentzianSimplexSolver.FourSimplexConnectivity.build_global_connectivity(simplices, geom)
    push!(geom.connectivity, conn)

    LorentzianSimplexSolver.FaceXiMatching.run_face_xi_matching(geom; sector=:ref)
    LorentzianSimplexSolver.GaugeFixingSU.run_su2_su11_gauge_fix(geom);
else
    sl2c = [geom.simplex[i].solgsl2c    for i in 1:ns]
    sgndet = [geom.simplex[i].sgndet    for i in 1:ns]
    geom.simplex[1].solgsl2c = LorentzianSimplexSolver.FaceXiMatching.update_sl2ctest(sl2c, sgndet)[1]
end;

In [5]:
# ------------------------------------------------------------
# 6. Define variables
# ------------------------------------------------------------
LorentzianSimplexSolver.DefineSymbols.run_define_variables(geom);

In [6]:
# ------------------------------------------------------------
# 7. Compute deficit angles, dihedral angles, and Regge action
# ------------------------------------------------------------
deficit_angles, dihedral_angles, _, _, iRegge = LorentzianSimplexSolver.ReggeAction.run_Regge_action(geom, simplices, vertex_coords);
iRegge

0.0 - 0.0962834958371431im

In [7]:
# ---------------------------------------------------------------------------------------------------------------
# 8. Compute the Spinfoam action at the critical point, here we already eliminate the boundary phase contribution
# ---------------------------------------------------------------------------------------------------------------
using SymEngine
γ = LorentzianSimplexSolver.DefineAction.γsym()
gamma_val = 1.0
sd, _ = LorentzianSimplexSolver.SolveVars.run_solver(geom);
vals = LorentzianSimplexSolver.ActionEvaluation.build_value_dict(sd, γ; γval=gamma_val);
S_no_phase, phase_soln = LorentzianSimplexSolver.SFaction_no_phase.compute_action_no_bdry_phase(geom, sd, dihedral_angles; γ=gamma_val);

S = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S_no_phase, phase_soln);
S_val = LorentzianSimplexSolver.ActionEvaluation.eval_symbolic(S, vals);
S_simpl = SymEngine.expand(S_val)

4.50161420078035e-15 - 0.0962834958361845*im

In [8]:
# ------------------------------------------------------------
# 8. Compute the eoms 
# ------------------------------------------------------------
dS = LorentzianSimplexSolver.EOMsHessian.compute_EOMs(S, sd)
dS = LorentzianSimplexSolver.EOMsHessian.check_EOMs(dS, sd; γ=gamma_val);

✔ All equations of motion satisfied (γ = 1.0, tol = 1.0e-8).


In [9]:
# # ----------------------------------------------------------------
# # 9. Compute the hessian matrix and the eigenvalues of the hessian 
# # ----------------------------------------------------------------
# g_vars = geom.varias[:g_var]
# z_vars = geom.varias[:z_var]
# η_vars = geom.varias[:η_var]
# xi_vars = geom.varias[:xi_var]
# vars = vcat(g_vars, z_vars, xi_vars, η_vars)
# H_sym = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S, vars)
# H_ref_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_sym, sd; γ = gamma_val);

# using LinearAlgebra
# eigenvalues = eigvals(H_ref_eval);
# Hess_eigenvals_sort = sort(eigenvalues, by=abs, rev=true);

In [10]:
# Hess_eigenvals_sort